# task_04 Solution

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

base = Path("../data")


In [ ]:

batches = pd.read_csv(base / "batches.csv")
maintenance = pd.read_csv(base / "maintenance.csv")
qc = pd.read_csv(base / "qc.csv")
machines = pd.read_csv(base / "machines.csv")

batches["batch_date"] = pd.to_datetime(batches["batch_date"])
maintenance["maintenance_date"] = pd.to_datetime(maintenance["maintenance_date"])

merged = batches.merge(qc, on="batch_id")

rows = []
for row in merged.itertuples(index=False):
    mm = maintenance[(maintenance["machine_id"] == row.machine_id) & (maintenance["maintenance_date"] <= row.batch_date)]
    if mm.empty:
        continue
    recent = mm["maintenance_date"].max()
    if (row.batch_date - recent).days <= 7:
        rows.append((row.machine_id, row.defect_units, row.units_produced))
post_maint = pd.DataFrame(rows, columns=["machine_id", "defect_units", "units_produced"])
defect_rate = post_maint.groupby("machine_id")["defect_units"].sum() / post_maint.groupby("machine_id")["units_produced"].sum()
q1 = str(defect_rate.sort_values(ascending=False).index[0])

line_b = merged.merge(machines[["machine_id", "line"]], on="machine_id")
line_b = line_b[line_b["line"] == "B"]
scrap_adj = (line_b["units_produced"] - line_b["defect_units"] - line_b["rework_units"]).sum() / line_b["units_produced"].sum()
q2 = f"{(scrap_adj * 100):.2f}%"

merged["good_units"] = merged["units_produced"] - merged["defect_units"]
merged["energy_per_good"] = merged["energy_kwh"] / merged["good_units"]
q3 = str(merged.sort_values(["energy_per_good", "batch_id"], ascending=[False, True]).iloc[0]["batch_id"])

post3_flags = []
for row in batches.itertuples(index=False):
    mm = maintenance[(maintenance["machine_id"] == row.machine_id) & (maintenance["maintenance_date"] <= row.batch_date)]
    if mm.empty:
        post3_flags.append(False)
    else:
        post3_flags.append((row.batch_date - mm["maintenance_date"].max()).days <= 3)
runtime_df = batches.copy()
runtime_df["post3"] = post3_flags
diff = runtime_df.loc[runtime_df["post3"], "runtime_hours"].mean() - runtime_df.loc[~runtime_df["post3"], "runtime_hours"].mean()
q4 = f"{diff:.2f}"

yields = batches.merge(qc, on="batch_id")
yields["yield_rate"] = (yields["units_produced"] - yields["defect_units"]) / yields["units_produced"]
yields["weekday"] = pd.to_datetime(yields["batch_date"]).dt.day_name()
q5 = str(yields.groupby("weekday")["yield_rate"].mean().sort_values(ascending=True).index[0])


Q1: Considering only batches produced within 7 days after the most recent maintenance on the same machine, which machine_id has the highest defect rate defined as total defect_units divided by total units_produced?

In [ ]:
print(q1)


Q2: What is the scrap-adjusted yield for line B, defined as (units_produced - defect_units - rework_units) / units_produced, expressed as a percentage rounded to 2 decimals?

In [ ]:
print(q2)


Q3: Which batch_id has the highest energy_kwh per good unit, where good units = units_produced - defect_units?

In [ ]:
print(q3)


Q4: What is the difference between mean runtime_hours for batches within 3 days after maintenance and mean runtime_hours for all other batches, rounded to 2 decimals?

In [ ]:
print(q4)


Q5: Which weekday has the lowest average yield rate, where yield rate = (units_produced - defect_units) / units_produced?

In [ ]:
print(q5)
